# NHANES Cardiometabolic Exploration

Reads the star-schema marts in `exports/` (built by `scripts/build_dataset.py`) and charts the peer-group outlier detection: distributions, anomaly breakdowns, and where undiagnosed outliers cluster.

Run `python scripts/build_dataset.py` first if `exports/` is empty. This notebook only reads — it doesn't re-download or re-process anything.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

EXPORT_DIR = Path("exports") if Path("exports").exists() else Path("../exports")

dim_respondents = pd.read_csv(EXPORT_DIR / "dim_respondents.csv")
fact_body = pd.read_csv(EXPORT_DIR / "fact_body_measures.csv")
fact_bp = pd.read_csv(EXPORT_DIR / "fact_blood_pressure.csv")
fact_labs = pd.read_csv(EXPORT_DIR / "fact_labs.csv")
fact_diagnoses = pd.read_csv(EXPORT_DIR / "fact_diagnoses.csv")
fact_anomalies = pd.read_csv(EXPORT_DIR / "fact_anomalies.csv")

# denormalized view for plotting — joins the star schema back together
wide = (
    dim_respondents
    .merge(fact_body, on="SEQN", how="left")
    .merge(fact_bp, on="SEQN", how="left")
    .merge(fact_labs, on="SEQN", how="left")
    .merge(fact_diagnoses, on="SEQN", how="left")
)

print(f"{len(wide):,} respondents, {len(fact_anomalies):,} flagged (respondent, field) pairs")

## Distributions by age band

BMI, blood pressure, and fasting glucose across the sample, so the peer-group z-score thresholds have visual context.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(
    axes,
    ["bmi", "mean_systolic", "fasting_glucose"],
    ["BMI", "Systolic BP (mmHg)", "Fasting Glucose (mg/dL)"],
):
    wide[col].dropna().hist(bins=40, ax=ax, color="steelblue")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Anomaly breakdown

Where the flagged (≥3 std dev from peer group) outliers concentrate — by field category and by age band.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

fact_anomalies["field_category"].value_counts().plot(
    kind="barh", ax=axes[0], color="crimson", title="Flagged outliers by category"
)
axes[0].invert_yaxis()

anomalies_with_age = fact_anomalies.merge(dim_respondents[["SEQN", "age_band"]], on="SEQN")
anomalies_with_age["age_band"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], color="darkorange", title="Flagged outliers by age band"
)

plt.tight_layout()
plt.show()

## Undiagnosed outliers

BMI vs. fasting glucose, colored by self-reported diabetes diagnosis — outliers with no diagnosis on record are the interesting cluster.

In [ ]:
plot_df = wide.dropna(subset=["bmi", "fasting_glucose"])
colors = plot_df["diabetes_diagnosis"].map({"Yes": "crimson", "No": "steelblue", "Borderline": "orange"}).fillna("gray")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot_df["bmi"], plot_df["fasting_glucose"], c=colors, alpha=0.4, s=12)
ax.set_xlabel("BMI")
ax.set_ylabel("Fasting Glucose (mg/dL)")
ax.set_title("BMI vs. fasting glucose, colored by diagnosed diabetes status")

handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=l)
           for l, c in [("Yes", "crimson"), ("No", "steelblue"), ("Borderline", "orange"), ("Not asked/unknown", "gray")]]
ax.legend(handles=handles, title="Diabetes diagnosis")
plt.show()

## Summary table

In [ ]:
summary = wide.groupby(["age_band", "gender"], observed=True).agg(
    respondents=("SEQN", "count"),
    avg_bmi=("bmi", "mean"),
    avg_systolic=("mean_systolic", "mean"),
    avg_glucose=("fasting_glucose", "mean"),
).round(1)
summary